# broadcast-initial-weights — worked example 3: Confirm a flattened state vector matches rank 0 on every replica

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcast-initial-weights`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A clean post-init check is to flatten every state tensor into one long vector and compare it against rank 0's vector. If broadcast was applied to all of `state_dict()`, every rank's flattened vector is bit-for-bit identical to rank 0's. This is the assertion a DDP unit test typically makes.

## Worked solution

**Goal.** Build the verification primitive used to test broadcast correctness.

1. **Flatten the whole state.** Concatenate every `state_dict()` tensor (reshaped to 1-D) into a single vector. This is a compact fingerprint of the model's entire state.
2. **Simulate three ranks.** Each builds a model with rank-specific init, so their pre-broadcast fingerprints all differ.
3. **Broadcast from rank 0.** Each replica copies rank 0's tensors in via the fake collective.
4. **Compare fingerprints.** After sync, every replica's flattened vector must equal rank 0's. We use `t.equal` for exactness — broadcast is a copy, not an average, so there is no floating-point tolerance to worry about.
5. **Why flatten works.** Because the module graph is identical across ranks, `state_dict()` yields tensors in the same order with the same shapes, so concatenation produces aligned vectors that are directly comparable.

In [ ]:
class FakeDist:
    def __init__(self, rank0_tensors):
        self.rank0_tensors = rank0_tensors
        self._i = 0
    def reset(self):
        self._i = 0
    def broadcast(self, tensor, src=0):
        tensor.copy_(self.rank0_tensors[self._i])
        self._i += 1

def build_model(rank):
    m = t.nn.Linear(2, 2, bias=True)
    with t.no_grad():
        m.weight.fill_(float(rank + 1))
        m.bias.fill_(float(-(rank + 1)))
    return m

def flat_state(model):
    return t.cat([v.reshape(-1).float() for v in model.state_dict().values()])

def broadcast_state(model, fake):
    fake.reset()
    for tensor in model.state_dict().values():
        fake.broadcast(tensor, src=0)

rank0 = build_model(0)
ref = flat_state(rank0)
fake = FakeDist([v.clone() for v in rank0.state_dict().values()])

results = []
for rank in [1, 2]:
    m = build_model(rank)
    before_match = t.equal(flat_state(m), ref)
    broadcast_state(m, fake)
    after_match = t.equal(flat_state(m), ref)
    results.append((rank, before_match, after_match))

for rank, b, a in results:
    print(f'rank {rank}: matched_before={b}  matched_after={a}')
print('all ranks synced:', all(a for _, _, a in results))